# Generative Adversarial Networks — Lab Notebook

This lab builds a GAN from scratch on MNIST and explores the key properties
that distinguish GANs from other generative models.

**What you will implement and explore:**
1. Vanilla GAN — Generator vs Discriminator, adversarial training loop
2. Training dynamics — loss curves, mode collapse, discriminator saturation
3. Generated samples — evolution across training epochs
4. Latent space interpolation — smooth walks through the generator
5. DCGAN — convolutional upgrade for sharper images
6. Conditional GAN (cGAN) — controlling which digit is generated
7. GAN vs VAE comparison — sharpness vs diversity

**Runtime:** use GPU (Runtime → Change runtime type → T4 GPU). Training takes ~3 min per model.

---
### Key GAN idea (recap)
A GAN trains two networks simultaneously:
- **Generator G**: maps random noise z ~ N(0,I) → fake image. Wants to fool D.
- **Discriminator D**: classifies real vs fake. Wants to catch G.

They play a minimax game until G produces images D cannot distinguish from real ones.

## 0. Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
import numpy as np
import os

DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'
LATENT_DIM = 100      # dimensionality of the noise vector z
IMG_SIZE   = 28 * 28  # flattened MNIST image
BATCH_SIZE = 128
EPOCHS     = 30

torch.manual_seed(42)
np.random.seed(42)
print(f'Device: {DEVICE}')

## 1. Data

In [ ]:
# Normalise to [-1, 1] — matches Tanh output of the generator
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))   # (x - 0.5) / 0.5  maps [0,1] → [-1,1]
])

train_data   = datasets.MNIST(root='data', train=True,  download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)

print(f'Training samples : {len(train_data)}')
print(f'Batches per epoch: {len(train_loader)}')

## 2. Vanilla GAN Architecture

### 2.1 Generator
Maps noise **z** ∈ ℝ¹⁰⁰ → fake image ∈ ℝ⁷⁸⁴.  
Key design choices:
- **LeakyReLU** hidden activations (lets small gradients through even for negative inputs)
- **Tanh** output (maps to [-1,1], matching the normalised real images)
- **BatchNorm** after each hidden layer (stabilises training)

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim=100, img_size=784):
        super().__init__()
        self.net = nn.Sequential(
            # Block 1: 100 → 256
            nn.Linear(latent_dim, 256),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(0.2, inplace=True),
            # Block 2: 256 → 512
            nn.Linear(256, 512),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.2, inplace=True),
            # Block 3: 512 → 1024
            nn.Linear(512, 1024),
            nn.BatchNorm1d(1024),
            nn.LeakyReLU(0.2, inplace=True),
            # Output: 1024 → 784, squash to [-1, 1]
            nn.Linear(1024, img_size),
            nn.Tanh()
        )

    def forward(self, z):
        return self.net(z).view(-1, 1, 28, 28)  # reshape to image


# Quick sanity check
G_test = Generator()
z_test = torch.randn(4, 100)
out    = G_test(z_test)
print(f'Generator output shape: {out.shape}')   # should be (4, 1, 28, 28)
print(f'Output range: [{out.min():.2f}, {out.max():.2f}]')  # should be in (-1, 1)

### 2.2 Discriminator
Maps image ∈ ℝ⁷⁸⁴ → scalar probability of being real.  
Key design choices:
- **LeakyReLU** (standard choice for discriminators — prevents dead neurons)
- **Dropout** (regularises the discriminator so it doesn't overpower the generator too early)
- **Sigmoid** output (probability ∈ [0,1])
- **No BatchNorm** on the discriminator input (recommended in the original paper)

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, img_size=784):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            # Block 1: 784 → 512
            nn.Linear(img_size, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),
            # Block 2: 512 → 256
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),
            # Output: 256 → 1, probability of being real
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, img):
        return self.net(img)   # shape: (B, 1)


D_test = Discriminator()
img_test = torch.randn(4, 1, 28, 28)
score    = D_test(img_test)
print(f'Discriminator output shape: {score.shape}')  # (4, 1)
print(f'Score range: [{score.min():.3f}, {score.max():.3f}]')  # (0, 1)

## 3. GAN Training Loop

The GAN trains two networks with **two separate optimisers** and **two separate loss steps** per batch:

**Step A — Train Discriminator:**
- Feed real images → D should output 1 (real)
- Feed fake images from G → D should output 0 (fake)
- Loss: `BCE(D(real), 1) + BCE(D(G(z)), 0)`
- Update **only D weights**

**Step B — Train Generator:**
- Feed new fake images → D should output 1 (generator wants to fool D)
- Loss: `BCE(D(G(z)), 1)`  — generator maximises D's mistake
- Update **only G weights**

**Critical:** G and D must be updated independently. Do not propagate G's gradient through D's update and vice versa.

In [ ]:
def train_gan(epochs=EPOCHS, lr=2e-4, latent_dim=LATENT_DIM):
    G = Generator(latent_dim).to(DEVICE)
    D = Discriminator().to(DEVICE)

    # Separate optimisers — each only updates its own network
    opt_G = torch.optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
    opt_D = torch.optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))

    criterion = nn.BCELoss()   # binary cross-entropy for real/fake labels

    # Fixed noise for visualisation (same z every epoch → tracks progress)
    fixed_z = torch.randn(64, latent_dim).to(DEVICE)

    history = {'d_loss': [], 'g_loss': [], 'd_real': [], 'd_fake': []}
    snapshots = {}   # generated images at selected epochs

    for epoch in range(1, epochs + 1):
        G.train(); D.train()
        ep_d, ep_g, ep_dr, ep_df = 0, 0, 0, 0

        for real_imgs, _ in train_loader:
            real_imgs = real_imgs.to(DEVICE)
            B = real_imgs.size(0)

            # Labels: 1 = real, 0 = fake
            real_labels = torch.ones(B, 1).to(DEVICE)
            fake_labels = torch.zeros(B, 1).to(DEVICE)

            # ── Step A: Train Discriminator ──────────────────────────────
            opt_D.zero_grad()

            # Real images → D should predict 1
            d_real = D(real_imgs)
            loss_real = criterion(d_real, real_labels)

            # Fake images → D should predict 0
            z = torch.randn(B, latent_dim).to(DEVICE)
            fake_imgs = G(z).detach()   # .detach() stops gradients flowing into G
            d_fake = D(fake_imgs)
            loss_fake = criterion(d_fake, fake_labels)

            loss_D = loss_real + loss_fake
            loss_D.backward()
            opt_D.step()

            # ── Step B: Train Generator ──────────────────────────────────
            opt_G.zero_grad()

            z = torch.randn(B, latent_dim).to(DEVICE)
            fake_imgs = G(z)
            # Generator wants D to predict real (1) for its fakes
            g_out = D(fake_imgs)
            loss_G = criterion(g_out, real_labels)   # fool the discriminator

            loss_G.backward()
            opt_G.step()

            # Track metrics
            ep_d  += loss_D.item();  ep_g  += loss_G.item()
            ep_dr += d_real.mean().item(); ep_df += d_fake.mean().item()

        n = len(train_loader)
        history['d_loss'].append(ep_d / n)
        history['g_loss'].append(ep_g / n)
        history['d_real'].append(ep_dr / n)
        history['d_fake'].append(ep_df / n)

        # Save generated images at key epochs
        if epoch in [1, 5, 10, 20, 30] or epoch == epochs:
            G.eval()
            with torch.no_grad():
                imgs = G(fixed_z).cpu()
            snapshots[epoch] = imgs
            G.train()

        print(f'Epoch {epoch:3d}/{epochs} | D: {ep_d/n:.3f}  G: {ep_g/n:.3f} '
              f'| D(real)={ep_dr/n:.3f}  D(fake)={ep_df/n:.3f}')

    return G, D, history, snapshots


print('Training Vanilla GAN ...')
G, D, history, snapshots = train_gan()

## 4. Training Dynamics

### Understanding the loss curves

- **D loss** ≈ log(2) ≈ 0.693 at equilibrium — D is guessing at chance level
- **G loss** high early (D easily catches fakes), should decrease as G improves
- **D(real)** = average score D assigns to real images (should stay near 0.5–0.7)
- **D(fake)** = average score D assigns to fake images (should rise as G improves)

**Warning signs:**
- D loss → 0: discriminator has won — G cannot fool it. Often means mode collapse.
- G loss → 0: unlikely, means G perfectly fools D (or D is broken).
- D(real) → 1 and D(fake) → 0 simultaneously: D has completely dominated G.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

epochs_range = range(1, len(history['d_loss']) + 1)

# Losses
axes[0].plot(epochs_range, history['d_loss'], label='D loss', color='crimson')
axes[0].plot(epochs_range, history['g_loss'], label='G loss', color='steelblue')
axes[0].axhline(0.693, color='gray', linestyle='--', lw=0.8, label='log(2) equilibrium')
axes[0].set_title('GAN losses'); axes[0].set_xlabel('Epoch')
axes[0].legend()

# D(real) and D(fake) — measure of balance
axes[1].plot(epochs_range, history['d_real'], label='D(real)', color='seagreen')
axes[1].plot(epochs_range, history['d_fake'], label='D(fake)', color='darkorange')
axes[1].axhline(0.5, color='gray', linestyle='--', lw=0.8, label='0.5 equilibrium')
axes[1].set_title('Discriminator scores'); axes[1].set_xlabel('Epoch')
axes[1].set_ylim(0, 1); axes[1].legend()

plt.tight_layout()
plt.savefig('gan_training_curves.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Generated Images — Evolution Across Training

The **same fixed noise vector** z is decoded at each snapshot epoch.  
This lets us see how the same "latent identity" improves across training.

In [ ]:
def show_snapshots(snapshots, n_col=8):
    epochs_shown = sorted(snapshots.keys())
    fig, axes = plt.subplots(len(epochs_shown), 1,
                             figsize=(n_col * 1.2, len(epochs_shown) * 1.4))
    for ax, ep in zip(axes, epochs_shown):
        imgs = snapshots[ep][:n_col]           # first n_col images
        grid = make_grid(imgs, nrow=n_col,
                         normalize=True,       # maps [-1,1] → [0,1] for display
                         value_range=(-1, 1))
        ax.imshow(grid.permute(1, 2, 0).squeeze(), cmap='gray')
        ax.set_ylabel(f'Epoch {ep}', fontsize=11)
        ax.axis('off')
    plt.suptitle('Generated samples (same z, different epochs)', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig('gan_evolution.png', dpi=120, bbox_inches='tight')
    plt.show()


show_snapshots(snapshots)

## 6. Final Generated Samples

In [ ]:
G.eval()
with torch.no_grad():
    z = torch.randn(64, LATENT_DIM).to(DEVICE)
    samples = G(z).cpu()

grid = make_grid(samples, nrow=8, normalize=True, value_range=(-1, 1))
fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(grid.permute(1, 2, 0).squeeze(), cmap='gray')
ax.axis('off')
ax.set_title('GAN — 64 generated samples', fontsize=14)
plt.tight_layout()
plt.savefig('gan_samples.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. Latent Space Interpolation

Unlike VAEs, GANs have no explicit encoder.  
But the **generator** still defines a smooth mapping z → image.  
We can interpolate between two noise vectors and see the transition.

In [ ]:
def interpolate_gan(G, z1, z2, steps=10):
    """Linear interpolation between two noise vectors."""
    G.eval()
    alphas = torch.linspace(0, 1, steps)
    imgs   = []
    with torch.no_grad():
        for a in alphas:
            z = (1 - a) * z1 + a * z2
            imgs.append(G(z.unsqueeze(0).to(DEVICE)).cpu().squeeze())
    return imgs


# 3 independent interpolations
STEPS = 12
fig, axes = plt.subplots(3, STEPS, figsize=(STEPS * 1.2, 4.5))

for row in range(3):
    z1 = torch.randn(LATENT_DIM)
    z2 = torch.randn(LATENT_DIM)
    interp = interpolate_gan(G, z1, z2, steps=STEPS)
    for col, img in enumerate(interp):
        axes[row, col].imshow(img.squeeze(), cmap='gray', vmin=-1, vmax=1)
        axes[row, col].axis('off')

plt.suptitle('GAN latent interpolation (z₁ → z₂)', fontsize=13)
plt.tight_layout()
plt.savefig('gan_interpolation.png', dpi=120, bbox_inches='tight')
plt.show()

## 8. DCGAN — Deep Convolutional GAN

The original GAN uses only linear layers.  
**DCGAN** (Radford et al., 2015) replaces them with convolutions:

- **Generator**: `ConvTranspose2d` (transposed convolution / "upsampling") layers to grow spatial resolution
- **Discriminator**: `Conv2d` layers to extract spatial features
- **BatchNorm** in both networks (except D input and G output)
- **No pooling** in the discriminator — strided convolutions instead
- Result: **sharper, more coherent** images

In [ ]:
def weights_init(m):
    """DCGAN weight initialisation: N(0, 0.02) for conv/linear, 1/0 for batchnorm."""
    cls = m.__class__.__name__
    if 'Conv' in cls or 'Linear' in cls:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif 'BatchNorm' in cls:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


class DCGenerator(nn.Module):
    """Generator using transposed convolutions. Input: (B, latent_dim, 1, 1)."""
    def __init__(self, latent_dim=100, ngf=64):
        super().__init__()
        self.net = nn.Sequential(
            # z: (B, latent_dim, 1, 1) → (B, ngf*4, 7, 7)
            nn.ConvTranspose2d(latent_dim, ngf * 4, 7, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 4), nn.ReLU(True),
            # (B, ngf*4, 7, 7) → (B, ngf*2, 14, 14)  stride=2 doubles spatial size
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2), nn.ReLU(True),
            # (B, ngf*2, 14, 14) → (B, 1, 28, 28)
            nn.ConvTranspose2d(ngf * 2, 1, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, z):
        return self.net(z.view(z.size(0), -1, 1, 1))


class DCDiscriminator(nn.Module):
    """Discriminator using strided convolutions (no pooling)."""
    def __init__(self, ndf=64):
        super().__init__()
        self.net = nn.Sequential(
            # (B, 1, 28, 28) → (B, ndf, 14, 14)  stride=2 halves spatial size
            nn.Conv2d(1, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # (B, ndf, 14, 14) → (B, ndf*2, 7, 7)
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2), nn.LeakyReLU(0.2, inplace=True),
            # (B, ndf*2, 7, 7) → (B, 1, 1, 1)
            nn.Conv2d(ndf * 2, 1, 7, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, img):
        return self.net(img).view(-1, 1)


def train_dcgan(epochs=EPOCHS, lr=2e-4, latent_dim=LATENT_DIM):
    G_dc = DCGenerator(latent_dim).to(DEVICE); G_dc.apply(weights_init)
    D_dc = DCDiscriminator().to(DEVICE);       D_dc.apply(weights_init)

    opt_G = torch.optim.Adam(G_dc.parameters(), lr=lr, betas=(0.5, 0.999))
    opt_D = torch.optim.Adam(D_dc.parameters(), lr=lr, betas=(0.5, 0.999))
    criterion = nn.BCELoss()
    fixed_z   = torch.randn(64, latent_dim).to(DEVICE)

    history_dc = {'d_loss': [], 'g_loss': []}
    snapshots_dc = {}

    for epoch in range(1, epochs + 1):
        G_dc.train(); D_dc.train()
        ep_d = ep_g = 0
        for real_imgs, _ in train_loader:
            real_imgs = real_imgs.to(DEVICE)
            B = real_imgs.size(0)
            real_labels = torch.ones(B, 1).to(DEVICE)
            fake_labels = torch.zeros(B, 1).to(DEVICE)

            opt_D.zero_grad()
            loss_D = (criterion(D_dc(real_imgs), real_labels) +
                      criterion(D_dc(G_dc(torch.randn(B, latent_dim)
                                .to(DEVICE)).detach()), fake_labels))
            loss_D.backward(); opt_D.step()

            opt_G.zero_grad()
            loss_G = criterion(D_dc(G_dc(torch.randn(B, latent_dim).to(DEVICE))),
                               real_labels)
            loss_G.backward(); opt_G.step()

            ep_d += loss_D.item(); ep_g += loss_G.item()

        n = len(train_loader)
        history_dc['d_loss'].append(ep_d / n)
        history_dc['g_loss'].append(ep_g / n)

        if epoch in [1, 5, 10, 20, 30] or epoch == epochs:
            G_dc.eval()
            with torch.no_grad():
                snapshots_dc[epoch] = G_dc(fixed_z).cpu()
            G_dc.train()

        print(f'DCGAN Epoch {epoch:3d} | D: {ep_d/n:.3f}  G: {ep_g/n:.3f}')

    return G_dc, D_dc, history_dc, snapshots_dc


print('Training DCGAN ...')
G_dc, D_dc, history_dc, snapshots_dc = train_dcgan()

In [ ]:
# Compare Vanilla GAN vs DCGAN samples
G.eval(); G_dc.eval()
with torch.no_grad():
    z = torch.randn(32, LATENT_DIM).to(DEVICE)
    van_samples = G(z).cpu()
    dc_samples  = G_dc(z).cpu()

fig, axes = plt.subplots(2, 1, figsize=(14, 5))
for ax, imgs, title in zip(axes,
                            [van_samples, dc_samples],
                            ['Vanilla GAN', 'DCGAN']):
    grid = make_grid(imgs, nrow=16, normalize=True, value_range=(-1, 1))
    ax.imshow(grid.permute(1, 2, 0).squeeze(), cmap='gray')
    ax.set_ylabel(title, fontsize=12); ax.axis('off')

plt.suptitle('Vanilla GAN vs DCGAN — same z, same epoch count', fontsize=13)
plt.tight_layout()
plt.savefig('gan_vs_dcgan.png', dpi=120, bbox_inches='tight')
plt.show()

## 9. Conditional GAN (cGAN)

A standard GAN generates digits randomly — we cannot control which digit comes out.  
A **conditional GAN** conditions both G and D on a class label y:

- **Generator**: receives `concat(z, embed(y))` → generates digit of class y
- **Discriminator**: receives `concat(img_features, embed(y))` → real/fake *given* class y

This lets us generate any digit on demand: `G(z, y=3)` always produces a 3.

In [ ]:
class CondGenerator(nn.Module):
    def __init__(self, latent_dim=100, n_classes=10, embed_dim=50, img_size=784):
        super().__init__()
        # Label embedding: integer class → dense vector
        self.label_emb = nn.Embedding(n_classes, embed_dim)
        self.net = nn.Sequential(
            nn.Linear(latent_dim + embed_dim, 256),
            nn.BatchNorm1d(256), nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 512),
            nn.BatchNorm1d(512), nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(512, 1024),
            nn.BatchNorm1d(1024), nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(1024, img_size),
            nn.Tanh()
        )

    def forward(self, z, labels):
        # Concatenate noise and label embedding
        c = self.label_emb(labels)         # (B, embed_dim)
        inp = torch.cat([z, c], dim=1)     # (B, latent_dim + embed_dim)
        return self.net(inp).view(-1, 1, 28, 28)


class CondDiscriminator(nn.Module):
    def __init__(self, n_classes=10, embed_dim=50, img_size=784):
        super().__init__()
        self.label_emb = nn.Embedding(n_classes, embed_dim)
        self.net = nn.Sequential(
            nn.Linear(img_size + embed_dim, 512),
            nn.LeakyReLU(0.2, inplace=True), nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace=True), nn.Dropout(0.3),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, img, labels):
        c   = self.label_emb(labels)                  # (B, embed_dim)
        inp = torch.cat([img.view(img.size(0), -1), c], dim=1)
        return self.net(inp)


def train_cgan(epochs=EPOCHS, lr=2e-4, latent_dim=LATENT_DIM):
    G_c = CondGenerator(latent_dim).to(DEVICE)
    D_c = CondDiscriminator().to(DEVICE)
    opt_G = torch.optim.Adam(G_c.parameters(), lr=lr, betas=(0.5, 0.999))
    opt_D = torch.optim.Adam(D_c.parameters(), lr=lr, betas=(0.5, 0.999))
    criterion = nn.BCELoss()

    # Fixed: one column per digit (0–9), 6 rows each
    fixed_z   = torch.randn(60, latent_dim).to(DEVICE)
    fixed_lbl = torch.arange(10).repeat(6).to(DEVICE)  # 0,1,...,9,0,1,...

    for epoch in range(1, epochs + 1):
        G_c.train(); D_c.train()
        ep_d = ep_g = 0
        for real_imgs, labels in train_loader:
            real_imgs = real_imgs.to(DEVICE)
            labels    = labels.to(DEVICE)
            B = real_imgs.size(0)
            real_labels = torch.ones(B, 1).to(DEVICE)
            fake_labels = torch.zeros(B, 1).to(DEVICE)

            # D step — pass real class labels to both real and fake
            opt_D.zero_grad()
            z      = torch.randn(B, latent_dim).to(DEVICE)
            fake_y = torch.randint(0, 10, (B,)).to(DEVICE)
            loss_D = (criterion(D_c(real_imgs, labels), real_labels) +
                      criterion(D_c(G_c(z, fake_y).detach(), fake_y), fake_labels))
            loss_D.backward(); opt_D.step()

            # G step
            opt_G.zero_grad()
            z      = torch.randn(B, latent_dim).to(DEVICE)
            fake_y = torch.randint(0, 10, (B,)).to(DEVICE)
            loss_G = criterion(D_c(G_c(z, fake_y), fake_y), real_labels)
            loss_G.backward(); opt_G.step()

            ep_d += loss_D.item(); ep_g += loss_G.item()

        n = len(train_loader)
        print(f'cGAN Epoch {epoch:3d} | D: {ep_d/n:.3f}  G: {ep_g/n:.3f}')

    return G_c, D_c, fixed_z, fixed_lbl


print('Training conditional GAN ...')
G_c, D_c, fixed_z_c, fixed_lbl_c = train_cgan()

In [ ]:
# Show: one column per digit, same z per row → label controls the digit
G_c.eval()
with torch.no_grad():
    cond_imgs = G_c(fixed_z_c, fixed_lbl_c).cpu()

grid = make_grid(cond_imgs, nrow=10, normalize=True, value_range=(-1, 1))
fig, ax = plt.subplots(figsize=(12, 7))
ax.imshow(grid.permute(1, 2, 0).squeeze(), cmap='gray')
ax.set_title('cGAN — each column is one digit class (0–9)', fontsize=13)
for i in range(10):
    ax.text(i * 30 + 10, -8, str(i), ha='center', fontsize=11, color='black')
ax.axis('off')
plt.tight_layout()
plt.savefig('cgan_samples.png', dpi=120, bbox_inches='tight')
plt.show()

## 10. Mode Collapse — What It Looks Like

**Mode collapse** is the most common GAN failure: the generator learns to produce
only a few (or one) type of output, ignoring the diversity of the training data.

We simulate it by training with a very high learning rate and no regularisation.

In [ ]:
def train_collapsed_gan(epochs=15, lr=5e-3):  # deliberately bad hyperparams
    G_bad = Generator(LATENT_DIM).to(DEVICE)
    D_bad = Discriminator().to(DEVICE)
    # High LR + no beta tuning → unstable, collapse-prone
    opt_G = torch.optim.Adam(G_bad.parameters(), lr=lr)
    opt_D = torch.optim.Adam(D_bad.parameters(), lr=lr)
    criterion = nn.BCELoss()

    fixed_z = torch.randn(64, LATENT_DIM).to(DEVICE)
    snapshots_bad = {}

    for epoch in range(1, epochs + 1):
        G_bad.train(); D_bad.train()
        for real_imgs, _ in train_loader:
            real_imgs = real_imgs.to(DEVICE)
            B = real_imgs.size(0)
            rl = torch.ones(B, 1).to(DEVICE); fl = torch.zeros(B, 1).to(DEVICE)
            opt_D.zero_grad()
            z = torch.randn(B, LATENT_DIM).to(DEVICE)
            (criterion(D_bad(real_imgs), rl) +
             criterion(D_bad(G_bad(z).detach()), fl)).backward()
            opt_D.step()
            opt_G.zero_grad()
            z = torch.randn(B, LATENT_DIM).to(DEVICE)
            criterion(D_bad(G_bad(z)), rl).backward()
            opt_G.step()

        if epoch in [1, 5, 10, 15]:
            G_bad.eval()
            with torch.no_grad():
                snapshots_bad[epoch] = G_bad(fixed_z).cpu()
            G_bad.train()

    return snapshots_bad


print('Simulating mode collapse ...')
snapshots_bad = train_collapsed_gan()
show_snapshots(snapshots_bad)

## 11. GAN vs VAE — Visual Comparison

**Key difference:** GANs produce **sharper** images but with **less diversity**.
VAEs produce **blurrier** images (due to the pixel-level reconstruction loss) but
with **more controlled, continuous** latent space.

Load the VAE from the previous lab (or retrain it) to compare side by side.

In [ ]:
# ── Retrain a small VAE for comparison ──────────────────────────────────────
class VAEEncoder(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()
        self.shared = nn.Sequential(nn.Flatten(), nn.Linear(784, 400), nn.ReLU())
        self.fc_mu  = nn.Linear(400, latent_dim)
        self.fc_lv  = nn.Linear(400, latent_dim)
    def forward(self, x):
        h = self.shared(x); return self.fc_mu(h), self.fc_lv(h)

class VAEDecoder(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 400), nn.ReLU(),
            nn.Linear(400, 784), nn.Sigmoid()
        )
    def forward(self, z): return self.net(z).view(-1, 1, 28, 28)

class VAE(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()
        self.enc = VAEEncoder(latent_dim)
        self.dec = VAEDecoder(latent_dim)
    def forward(self, x):
        mu, lv = self.enc(x)
        z = mu + torch.exp(0.5*lv) * torch.randn_like(mu)
        return self.dec(z), mu, lv

# Normalise VAE data back to [0,1]
vae_transform = transforms.ToTensor()
vae_data   = datasets.MNIST(root='data', train=True, download=True, transform=vae_transform)
vae_loader = DataLoader(vae_data, batch_size=128, shuffle=True)

vae = VAE(latent_dim=100).to(DEVICE)   # use latent_dim=100 to match GAN
opt_vae = torch.optim.Adam(vae.parameters(), lr=1e-3)

print('Training VAE for comparison ...')
for epoch in range(1, 11):
    for x, _ in vae_loader:
        x = x.to(DEVICE)
        opt_vae.zero_grad()
        xhat, mu, lv = vae(x)
        recon = F.binary_cross_entropy(xhat, x, reduction='sum') / x.size(0)
        kl    = -0.5 * torch.sum(1 + lv - mu.pow(2) - lv.exp()) / x.size(0)
        (recon + kl).backward()
        opt_vae.step()
    print(f'  VAE epoch {epoch}')

# Compare
G.eval(); vae.eval()
with torch.no_grad():
    z_gan = torch.randn(32, LATENT_DIM).to(DEVICE)
    z_vae = torch.randn(32, 100).to(DEVICE)
    gan_imgs = G(z_gan).cpu()
    vae_imgs = vae.dec(z_vae).cpu()

fig, axes = plt.subplots(2, 1, figsize=(14, 5))
for ax, imgs, title in zip(axes,
                            [gan_imgs, vae_imgs],
                            ['GAN (implicit, sharp)', 'VAE (explicit p(x), blurry)']):
    grid = make_grid(imgs, nrow=16, normalize=True, value_range=(-1,1) if 'GAN' in title else (0,1))
    ax.imshow(grid.permute(1, 2, 0).squeeze(), cmap='gray')
    ax.set_ylabel(title, fontsize=11); ax.axis('off')

plt.suptitle('GAN vs VAE — sharpness vs diversity', fontsize=13)
plt.tight_layout()
plt.savefig('gan_vs_vae.png', dpi=120, bbox_inches='tight')
plt.show()

## 12. Summary

| Property | Vanilla GAN | DCGAN | cGAN | VAE |
|---|---|---|---|---|
| Architecture | MLP | Conv/ConvTranspose | MLP + label embed | Encoder + Decoder |
| Explicit p(x)? | No (implicit) | No | No | Yes |
| Controllable output? | No | No | Yes (by label) | Partially |
| Sample sharpness | Medium | High | Medium–High | Low (blurry) |
| Training stability | Fragile | More stable | Fragile | Stable |
| Mode collapse risk | High | Medium | Medium | Low |
| Latent space structure | Unstructured | Unstructured | Partially | Gaussian (KL) |

**Key takeaways:**
1. **Adversarial training** forces G to produce realistic images without any explicit reconstruction loss — this is why GAN samples can be sharper than VAE samples.
2. **`.detach()`** is critical — without it, discriminator gradients would flow into the generator during the D update, corrupting training.
3. **Separate optimisers** for G and D are essential — they are updated independently and should not share gradient information.
4. **D(real) ≈ D(fake) ≈ 0.5** at equilibrium — if D collapses to 0 or 1, training has failed.
5. **Mode collapse** is the main failure mode — the generator finds a few samples that fool D and stops diversifying.

In [ ]:
# Exercises
# 1. What happens if you remove .detach() from fake_imgs in the D update?
#    (Hint: try it and watch the G loss)
#
# 2. Try training with only 1 D step per 5 G steps (or vice versa).
#    Does balance matter?
#
# 3. In the cGAN, fix z and vary the label from 0 to 9.
#    Does the same z produce the same 'style' of each digit?
#
# 4. Implement label smoothing: replace real_labels = 1.0 with 0.9.
#    Does it reduce mode collapse?
#
# 5. Try the CelebA dataset with DCGAN. What features does the latent space encode?